# BıRAG — gelişme raporu için BLEU/ROUGE ve gecikme ölçümü

Başvuru formu İP2'de iki ölçüt var ki mevcut değerlendirme çıktılarıyla hesaplanamıyor:

| ölçüt (form) | neden hesaplanamıyordu | bu defter ne yapar |
|---|---|---|
| Türkçe testlerde **BLEU / ROUGE** > %80 | eval setleri referanssız (doğru cevap yazılı değil) | eğitimde görülmemiş **200 referanslı kayda** (ölçek-5k) iki modelle cevap üretir, **BERTScore**'u hesaplar |
| Ortalama **yanıt süresi** < 2 sn | eski süreler 8'lik partinin süresi/8 — tek istek değil | `golden.test`'in 48 sorusunu **tek tek** sorar; ilk jeton, görünen cevabın başlaması ve toplam süre |

Defter yalnız **üretir ve ölçer**. BLEU/chrF/ROUGE/F1 puanlaması ve rapor yerelde:
`scripts/analiz/2026-10-20-rapor-bleu-gecikme-puanla.py` (Drive'daki `birag-rapor-olcum` klasörünü repo köküne indirin).

Süre: A100 80 GB'de yaklaşık 1–1,5 saat (iki model × 200 üretim + 3 gecikme kolu × 48 istek).
Oturum koparsa yeniden çalıştırın: kaydedilmiş dosyalar atlanır.

# 0. Hazırlık (her oturumda)

In [ ]:
from google.colab import drive
drive.mount("/content/drive")
!pip install -q unsloth bert-score

In [ ]:
MODELLER = {
    "taban": "unsloth/gemma-4-26B-A4B-it",
    "ince":  "yildirim34/birag-gemma4-26b",
}
REFERANS = "data/eval-ref/olcek5k-ref.v1.jsonl"     # 200 kayıt — scripts/analiz/2026-10-20-rapor-referans-seti.py
REFERANS_SHA16 = "d8cce150543a3470"                # yereldeki dosyayla aynı mı — aşağıda denetlenir
GECIKME_SETI = "golden.test"                        # 48 terapötik soru
GECIKME_KOLLARI = [                                 # (model, düşünme açık mı)
    ("taban", True),
    ("ince", True),
    ("ince", False),                                # düşünmesiz: gecikmenin ne kadarı düşünmeden geliyor
]
ISINMA = 2                                          # ilk 2 istek ölçülmez (CUDA ısınması)
CIKTI = "/content/drive/MyDrive/birag-rapor-olcum"
MAX_YENI_JETON = 2048                               # önceki değerlendirmeyle aynı
BATCH = 8                                           # yalnız referans üretiminde; gecikme hep tek istek
BERT_MODEL = "bert-base-multilingual-cased"         # bert-score'un Türkçe için varsayılanı

In [ ]:
import json, os, time, hashlib
if not os.path.exists("/content/birag"):
    !git clone -q --depth 1 https://github.com/enveryildirim/birag-data-generation.git /content/birag

def jsonl_oku(yol):
    return [json.loads(x) for x in open(yol, encoding="utf-8") if x.strip()] if os.path.exists(yol) else []

def jsonl_yaz(yol, satirlar):
    os.makedirs(os.path.dirname(yol), exist_ok=True)
    with open(yol, "w", encoding="utf-8") as f:
        for r in satirlar: f.write(json.dumps(r, ensure_ascii=False) + "\n")

ref_yol = f"/content/birag/{REFERANS}"
sha = hashlib.sha256(open(ref_yol, "rb").read()).hexdigest()[:16]
assert sha == REFERANS_SHA16, f"⛔ referans dosyası farklı: {sha} — repo güncel mi?"
REF = jsonl_oku(ref_yol)
GEC = jsonl_oku(f"/content/birag/evals/{GECIKME_SETI}.jsonl")
print(f"referans {len(REF)} kayıt · gecikme seti {len(GEC)} soru · sha {sha} ✅")

In [ ]:
# Önceki değerlendirme defteriyle (taban-vs-ince-basit) AYNI yükleme ve üretim
from unsloth import FastModel
import torch, gc

def yukle(repo):
    model, tokenizer = FastModel.from_pretrained(repo, max_seq_length=8192, dtype=torch.bfloat16, load_in_4bit=False)
    FastModel.for_inference(model)
    tok = getattr(tokenizer, "tokenizer", tokenizer)
    tok.padding_side = "left"
    if tok.pad_token is None: tok.pad_token = tok.eos_token
    return model, tok

def bosalt():
    for ad in ("model", "tok"):
        globals().pop(ad, None)
    gc.collect(); torch.cuda.empty_cache()

def ayir(ham):
    # "<|channel>thought ... <channel|>cevap"  →  (düşünce, cevap)
    if "<|channel>" not in ham:
        return "", ham.strip()
    if "<channel|>" not in ham:                       # düşünürken kesilmiş
        return ham.split("<|channel>", 1)[1].removeprefix("thought").strip(), ""
    d, c = ham.split("<channel|>", 1)
    return d.split("<|channel>", 1)[1].removeprefix("thought").strip(), c.strip()

def istem(tok, mesajlar, dusunme=True):
    return tok.apply_chat_template([{"role": m["role"], "content": m["content"]} for m in mesajlar], tokenize=False,
                                   add_generation_prompt=True, enable_thinking=dusunme)

def uret(model, tok, mesaj_listesi):
    x = tok([istem(tok, ms) for ms in mesaj_listesi], return_tensors="pt", padding=True, add_special_tokens=False).to("cuda")
    t0 = time.time()
    with torch.no_grad():
        y = model.generate(**x, max_new_tokens=MAX_YENI_JETON, do_sample=False, pad_token_id=tok.pad_token_id)
    sure = (time.time() - t0) / len(mesaj_listesi)
    sonuc = []
    for satir in y[:, x["input_ids"].shape[1]:].tolist():
        satir = [t for t in satir if t != tok.pad_token_id]
        ham = tok.decode(satir, skip_special_tokens=False)
        dusunce, cevap = ayir(ham.split("<turn|>")[0])
        sonuc.append({"dusunce": dusunce, "cevap": cevap, "kesildi": "<turn|>" not in ham,
                      "jeton": len(satir), "sure_sn": round(sure, 1)})
    return sonuc

In [ ]:
# Gecikme: TEK istek, akışla. Üç an ölçülür:
#   ilk_jeton_sn   — model ilk jetonu ne zaman verdi (düşünce dahil)
#   cevap_bas_sn   — kullanıcının göreceği cevabın ilk karakteri ne zaman geldi (düşünce bittikten sonra)
#   toplam_sn      — üretim ne zaman bitti
from transformers import TextIteratorStreamer
from threading import Thread

def tek_istek(model, tok, mesajlar, dusunme):
    x = tok([istem(tok, mesajlar, dusunme)], return_tensors="pt", add_special_tokens=False).to("cuda")
    akis = TextIteratorStreamer(tok, skip_prompt=True, skip_special_tokens=False)
    is_ = Thread(target=model.generate, kwargs=dict(**x, max_new_tokens=MAX_YENI_JETON, do_sample=False,
                                                     streamer=akis, pad_token_id=tok.pad_token_id))
    torch.cuda.synchronize(); t0 = time.time()
    is_.start()
    metin, ilk, cevap_bas = "", None, None
    for parca in akis:
        if ilk is None and parca: ilk = time.time() - t0
        metin += parca
        if cevap_bas is None:
            if "<channel|>" in metin: gorunen = metin.split("<channel|>", 1)[1]
            elif "<|channel>" in metin: gorunen = ""          # hâlâ düşünüyor
            else: gorunen = metin
            if gorunen.replace("<turn|>", "").strip(): cevap_bas = time.time() - t0
    is_.join(); torch.cuda.synchronize(); toplam = time.time() - t0
    dusunce, cevap = ayir(metin.split("<turn|>")[0])
    return {"ilk_jeton_sn": round(ilk or toplam, 3), "cevap_bas_sn": round(cevap_bas or toplam, 3),
            "toplam_sn": round(toplam, 3), "jeton": len(tok(metin, add_special_tokens=False)["input_ids"]),
            "kesildi": "<turn|>" not in metin, "dusunce": dusunce, "cevap": cevap}

# 1. Üretim ve gecikme
Her model **bir kez** yüklenir: önce 200 referans kaydına cevap (parti 8), sonra o modelin gecikme kolları (tek istek).
Çıktılar: `CIKTI/<model>/referans.jsonl` · `CIKTI/<model>/bilgi.json` · `CIKTI/gecikme/<model>-<dusunme|dusunmesiz>.jsonl`.
İlk cevap ekrana basılır — **düşünce boşsa ya da cevap kesilmişse durun.**

In [ ]:
from huggingface_hub import model_info

for ad, repo in MODELLER.items():
    ref_yol = f"{CIKTI}/{ad}/referans.jsonl"
    kollar = [d for m, d in GECIKME_KOLLARI if m == ad]
    gec_yol = lambda d: f"{CIKTI}/gecikme/{ad}-{'dusunme' if d else 'dusunmesiz'}.jsonl"
    eksik_kol = [d for d in kollar if not os.path.exists(gec_yol(d))]
    if os.path.exists(ref_yol) and not eksik_kol:
        print(f"✅ {ad}: hepsi kaydedilmiş"); continue
    print(f"══ {ad}: {repo} yükleniyor")
    model, tok = yukle(repo)
    os.makedirs(f"{CIKTI}/{ad}", exist_ok=True)
    json.dump({"repo": repo, "revizyon": model_info(repo).sha, "gpu": torch.cuda.get_device_name(0),
               "torch": torch.__version__, "max_yeni_jeton": MAX_YENI_JETON, "batch": BATCH,
               "referans_sha16": REFERANS_SHA16, "tarih": time.strftime("%Y-%m-%d %H:%M")},
              open(f"{CIKTI}/{ad}/bilgi.json", "w"), indent=1)

    if not os.path.exists(ref_yol):
        satirlar, t0 = [], time.time()
        for i in range(0, len(REF), BATCH):
            parca = REF[i:i + BATCH]
            for o, r in zip(parca, uret(model, tok, [o["messages"] for o in parca])):
                satirlar.append({"id": o["id"], **r})
            if i == 0:
                print("  ilk düşünce:", satirlar[0]["dusunce"][:200].replace("\n", " "))
                print("  ilk cevap  :", satirlar[0]["cevap"][:200].replace("\n", " "))
        jsonl_yaz(ref_yol, satirlar)
        print(f"  referans: {len(satirlar)} cevap · {(time.time() - t0) / 60:.1f} dk")

    for d in eksik_kol:
        for o in GEC[:ISINMA]:
            tek_istek(model, tok, o["messages"], d)          # ısınma, kaydedilmez
        satirlar = [{"id": o["id"], "dusunme": d, **tek_istek(model, tok, o["messages"], d)} for o in GEC]
        jsonl_yaz(gec_yol(d), satirlar)
        ort = sum(r["toplam_sn"] for r in satirlar) / len(satirlar)
        print(f"  gecikme ({'düşünmeli' if d else 'düşünmesiz'}): {len(satirlar)} istek · toplam ort {ort:.1f} sn")
    bosalt()

# 2. BERTScore
Cevapla referansın **anlam** benzerliği (sözcük örtüşmesi değil). Model: `bert-base-multilingual-cased`.
Boş cevap 0 puan alır (kesilip cevap veremeyen model ödüllendirilmesin). Çıktı: `CIKTI/bertscore.jsonl`.

In [ ]:
from bert_score import score
import bert_score

BS_YOL = f"{CIKTI}/bertscore.jsonl"
if os.path.exists(BS_YOL):
    print("✅ bertscore kaydedilmiş")
else:
    ref = {o["id"]: o["referans"] for o in REF}
    satirlar = []
    for ad in MODELLER:
        cev = jsonl_oku(f"{CIKTI}/{ad}/referans.jsonl")
        dolu = [r for r in cev if r["cevap"].strip()]
        P, R, F = score([r["cevap"] for r in dolu], [ref[r["id"]] for r in dolu], model_type=BERT_MODEL,
                        lang="tr", batch_size=32, verbose=False)
        puan = {r["id"]: (p, rr, f) for r, p, rr, f in zip(dolu, P.tolist(), R.tolist(), F.tolist())}
        for r in cev:
            p, rr, f = puan.get(r["id"], (0.0, 0.0, 0.0))
            satirlar.append({"model": ad, "id": r["id"], "P": round(p, 4), "R": round(rr, 4), "F": round(f, 4),
                             "bos_cevap": not r["cevap"].strip()})
        print(f"  {ad}: {len(dolu)}/{len(cev)} dolu cevap puanlandı")
    jsonl_yaz(BS_YOL, satirlar)
    json.dump({"bert_model": BERT_MODEL, "bert_score": bert_score.__version__, "tarih": time.strftime("%Y-%m-%d %H:%M")},
              open(f"{CIKTI}/bertscore_bilgi.json", "w"), indent=1)
    print("✅ kaydedildi")

# 3. Bitti
Drive'daki **`birag-rapor-olcum`** klasörünü indirip repo köküne koyun (`birag-degerlendirme/` gibi). Puanlama ve rapor yerelde:

```
uv run --with sacrebleu --with rouge-score python scripts/analiz/2026-10-20-rapor-bleu-gecikme-puanla.py
```